# Component 3 — Response Simulator

The simulator takes a (persona, image features) pair and outputs a noisy engagement score.

## How it works
1. Convert image features to a binary/normalized numeric vector
2. Dot product with the persona's preference weights → raw score
3. Scale into [0, 1] range: `(raw + 3) / 6`
4. Add Gaussian noise (mean=0, std=0.3)
5. Clip to [0, 1]

## Noise Model
We use additive Gaussian noise with **σ = 0.3**. This means:
- ~68% of scores fall within ±0.3 of the true value
- ~95% fall within ±0.6
- The bandit needs roughly 20-50 observations per image to reliably distinguish good from bad

This calibration ensures the problem is non-trivial (can't solve in one pass) but learnable (doesn't require thousands of samples).

In [ ]:
import json
import numpy as np
from simulator import score, featurize

with open("image_features.json") as f:
    image_features = json.load(f)

with open("personas.json") as f:
    personas = json.load(f)["personas"]

In [ ]:
# Show what featurization looks like for one image
example_name = list(image_features.keys())[0]
print(f"Raw features for {example_name}:")
print(json.dumps(image_features[example_name], indent=2))
print(f"\nNumeric vector:")
for k, v in featurize(image_features[example_name]).items():
    print(f"  {k:25s} {v:.1f}")

In [ ]:
# Sanity check: top 3 images per persona (averaged over 50 samples to smooth noise)
rng = np.random.default_rng(42)

for persona in personas:
    scores = {}
    for filename, features in image_features.items():
        if "_parse_error" in features:
            continue
        avg = np.mean([score(persona, features, rng) for _ in range(50)])
        scores[filename] = avg

    ranked = sorted(scores.items(), key=lambda x: -x[1])
    print(f"\n{persona['name']} ({persona['background'][:60]}...)")
    for fname, s in ranked[:3]:
        print(f"  {s:.3f}  {fname}")

In [ ]:
# Demonstrate noise: score the same image 20 times for one persona
import matplotlib.pyplot as plt

persona = personas[0]  # Eleanor
img = image_features[list(image_features.keys())[0]]
rng = np.random.default_rng(42)

samples = [score(persona, img, rng) for _ in range(200)]

plt.hist(samples, bins=20, edgecolor='black')
plt.title(f"Score distribution: {persona['name']} × {list(image_features.keys())[0]}")
plt.xlabel("Engagement Score")
plt.ylabel("Count")
plt.axvline(np.mean(samples), color='red', linestyle='--', label=f'mean={np.mean(samples):.3f}')
plt.legend()
plt.show()

print(f"Mean: {np.mean(samples):.3f}, Std: {np.std(samples):.3f}")